In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
#URL_FULL = "http://58.186.149.100:19001/"
#URL_QUANTIZED = "http://58.186.149.100:19002/"
URL_FULL = "http://0.0.0.0:8010"
URL_QUANTIZED = "http://0.0.0.0:8005"
MAX_TOKENS = 3000
TEMPERATURE = 0.99
SEED = 42
LOGPROBS = 4

import sys
sys.path.append('../src')
sys.path.append('../../common/src')

from validation.prompts import get_squad_data_questions
from validation.runner import run_validation
from validation.data import (
    ModelInfo,
    RequestParams,
    save_to_jsonl
)


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

In [8]:
from datasets import load_dataset
from typing import List


def get_squad_data_questions() -> List[str]:
    dataset = load_dataset('squad', keep_in_memory=True)
    prompts = []
    
    train_prompts = [f"Context: {context}\nQuestion: {question} " for question, context in zip(dataset['train']['question'], dataset['train']['context'])]
    prompts.extend(train_prompts)
    
    validation_prompts = [f"Context: {context}\nQuestion: {question} " for question, context in zip(dataset['validation']['question'], dataset['validation']['context'])]
    prompts.extend(validation_prompts)
    
    return prompts

prompts = get_squad_data_questions()

In [9]:
full_model_info = ModelInfo(
    url=URL_FULL,
    name="Qwen/Qwen3-0.6B",
    deploy_params={
        "GPU": "0.3xRTX4000",
        "precision": "fp16",
    }
)

quantized_model_info = ModelInfo(
    url=URL_QUANTIZED,
    name="Qwen/Qwen3-0.6B",
    deploy_params={
        "GPU": "0.3xRTX4000",
        "precision": "fp8",
    }
)

request_params = RequestParams(
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
    top_logprobs=LOGPROBS
)

inference_model_info = quantized_model_info
validation_model_info = full_model_info

In [ ]:
DATA_PATH = 'qwen3-0.6B_fp16_val_fp16.jsonl'

batch_size = 500

prompts = prompts

for start_idx in range(0, len(prompts), batch_size):
    prompt_batch = prompts[start_idx:start_idx + batch_size]
    results_batch = run_validation(
        prompt_batch,
        inference_model=inference_model_info,
        validation_model=validation_model_info,
        request_params=request_params,
        max_workers=50
    )
    save_to_jsonl(results_batch, DATA_PATH, append=True)
    print(f"Processed {start_idx + batch_size} from {len(prompts)}")

Processed 500 from 98169


Processed 1000 from 98169


Processed 1500 from 98169


Processed 2000 from 98169


Processed 2500 from 98169


...

Processed 16500 from 98169


Processed 17000 from 98169


Processed 17500 from 98169


Processed 18000 from 98169


Processed 18500 from 98169
